# Generate XAI Maps for Halo Mass Regression

Generate Grad-CAM and Integrated Gradients maps for a trained halo-mass regression checkpoint.

**Important note:** Attribution maps show regions influential for the model prediction, not direct dark matter detections.

## 1. Setup Environment

In [ ]:
# Configure Google Colab environment when needed
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive', force_remount=False)
    project_path = Path('/content/drive/MyDrive/xai-dark-matter-localization')
    os.chdir(project_path)
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
else:
    project_path = Path.cwd()

print(f'Working directory: {Path.cwd()}')

## 2. Load Config and Model

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import colormaps
from torchvision import transforms

from src.data.image_dataset import HaloMassImageDataset
from src.training.evaluate_regression import load_yaml_config, get_device, load_checkpoint_model
from src.models.halo_mass_regressor import predict_scalar
from src.explainability.gradcam import GradCAM
from src.explainability.integrated_gradients import integrated_gradients, channel_reduced_ig_map

config_path = Path('configs/regression_resnet18.yaml')
config = load_yaml_config(config_path)
device = get_device(config.get('device', 'auto'))

# If your checkpoint is somewhere else in Drive, set this manually, for example:
# CHECKPOINT_PATH = Path('/content/drive/MyDrive/xai-dark-matter-localization/outputs/regression_resnet18/checkpoints/best_model.pt')
CHECKPOINT_PATH = None

def find_checkpoint(config):
    output_dir = Path(config['output']['output_dir'])
    candidates = []
    if CHECKPOINT_PATH is not None:
        candidates.append(Path(CHECKPOINT_PATH))
    candidates.extend([
        output_dir / 'checkpoints' / 'best_model.pt',
        output_dir / 'checkpoints' / 'last_model.pt',
        Path('outputs/regression_resnet18/checkpoints/best_model.pt'),
        Path('outputs/regression_resnet18/checkpoints/last_model.pt'),
        Path('/content/drive/MyDrive/xai-dark-matter-localization/outputs/regression_resnet18/checkpoints/best_model.pt'),
        Path('/content/drive/MyDrive/xai-dark-matter-localization/outputs/regression_resnet18/checkpoints/last_model.pt'),
    ])
    for candidate in candidates:
        if candidate.exists():
            return candidate

    discovered = sorted(Path('.').glob('**/best_model.pt')) + sorted(Path('.').glob('**/last_model.pt'))
    if discovered:
        return discovered[0]

    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(
        'No trained checkpoint was found. Run notebooks/06_train_halo_mass_regressor.ipynb first, '
        'or set CHECKPOINT_PATH in this cell to the exact .pt file.\n\nChecked:\n' + checked
    )

checkpoint_path = find_checkpoint(config)
model = load_checkpoint_model(config, checkpoint_path, device)
print(f'Device: {device}')
print(f'Checkpoint: {checkpoint_path}')

## 3. Build Test Dataset

The target label is used only for reporting. Attribution is computed from the model prediction.

In [ ]:
data_config = config['data']
image_size = int(data_config.get('image_size', 224))

xai_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = HaloMassImageDataset(
    csv_path=data_config['metadata_csv'],
    split='test',
    root_dir=data_config.get('root_dir', '.'),
    transform=xai_transform,
    target_column=data_config.get('target_column', 'halo_mass_log10'),
    image_mode=data_config.get('image_mode', 'RGB'),
    return_metadata=True,
)

print(f'Test samples available: {len(test_dataset)}')

## 4. Helper Functions

In [ ]:
RESULTS_DIR = Path('results/xai')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NOTE = 'Attribution maps show regions influential for the model prediction, not direct dark matter detections.'
print(NOTE)

def tensor_to_numpy_map(tensor):
    return tensor.detach().cpu().numpy().astype(np.float32)

def load_resized_original(image_path, image_size):
    path = Path(image_path)
    if not path.is_absolute():
        path = Path.cwd() / path
    image = Image.open(path).convert('RGB').resize((image_size, image_size))
    return np.asarray(image, dtype=np.float32) / 255.0

def save_heatmap(map_array, output_path, cmap_name='magma'):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.imsave(output_path, map_array, cmap=cmap_name)

def save_overlay(original_rgb, map_array, output_path, alpha=0.45, cmap_name='magma'):
    cmap = colormaps[cmap_name]
    heat_rgb = cmap(map_array)[..., :3]
    overlay = np.clip((1 - alpha) * original_rgb + alpha * heat_rgb, 0, 1)
    plt.imsave(output_path, overlay)

def sample_output_stem(method, index, subhalo_id):
    return f'{method}_idx{index:04d}_subhalo{subhalo_id}'

## 5. Generate Grad-CAM and Integrated Gradients

In [ ]:
num_samples = min(8, len(test_dataset))
ig_steps = 50
np.random.seed(int(config.get('seed', 42)))
sample_indices = np.random.choice(len(test_dataset), size=num_samples, replace=False)

gradcam = GradCAM(model)
summary_rows = []

try:
    for index in sample_indices:
        image_tensor, target, metadata = test_dataset[int(index)]
        image_batch = image_tensor.unsqueeze(0).to(device)
        image_batch.requires_grad_(True)

        with torch.no_grad():
            prediction = float(predict_scalar(model, image_batch).detach().cpu().item())

        subhalo_id = metadata['subhalo_id']
        true_value = float(target)
        original = load_resized_original(metadata['image_path'], image_size)

        cam_map = tensor_to_numpy_map(gradcam(image_batch))
        stem = sample_output_stem('gradcam', int(index), subhalo_id)
        np.save(RESULTS_DIR / f'{stem}.npy', cam_map)
        save_heatmap(cam_map, RESULTS_DIR / f'{stem}_heatmap.png')
        save_overlay(original, cam_map, RESULTS_DIR / f'{stem}_overlay.png')

        image_batch_for_ig = image_tensor.unsqueeze(0).to(device)
        ig_attr = integrated_gradients(model, image_batch_for_ig, steps=ig_steps)
        ig_raw = tensor_to_numpy_map(ig_attr)
        ig_map = tensor_to_numpy_map(channel_reduced_ig_map(ig_attr))
        stem = sample_output_stem('integrated_gradients', int(index), subhalo_id)
        np.save(RESULTS_DIR / f'{stem}_raw.npy', ig_raw)
        np.save(RESULTS_DIR / f'{stem}_map.npy', ig_map)
        save_heatmap(ig_map, RESULTS_DIR / f'{stem}_heatmap.png')
        save_overlay(original, ig_map, RESULTS_DIR / f'{stem}_overlay.png')

        summary_rows.append({
            'dataset_index': int(index),
            'subhalo_id': subhalo_id,
            'image_path': metadata['image_path'],
            'true_halo_mass_log10': true_value,
            'pred_halo_mass_log10': prediction,
            'gradcam_npy': str(RESULTS_DIR / f'gradcam_idx{int(index):04d}_subhalo{subhalo_id}.npy'),
            'integrated_gradients_raw_npy': str(RESULTS_DIR / f'integrated_gradients_idx{int(index):04d}_subhalo{subhalo_id}_raw.npy'),
            'note': NOTE,
        })
finally:
    gradcam.close()

summary_df = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / 'xai_summary.csv'
summary_df.to_csv(summary_path, index=False)
summary_df

## 6. Preview One Result

In [ ]:
if len(summary_df) > 0:
    row = summary_df.iloc[0]
    subhalo_id = row['subhalo_id']
    index = int(row['dataset_index'])
    original = load_resized_original(row['image_path'], image_size)
    cam_map = np.load(RESULTS_DIR / f'gradcam_idx{index:04d}_subhalo{subhalo_id}.npy')
    ig_map = np.load(RESULTS_DIR / f'integrated_gradients_idx{index:04d}_subhalo{subhalo_id}_map.npy')

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(original)
    axes[0].set_title('Input image')
    axes[1].imshow(original)
    axes[1].imshow(cam_map, cmap='magma', alpha=0.45)
    axes[1].set_title('Grad-CAM overlay')
    axes[2].imshow(original)
    axes[2].imshow(ig_map, cmap='magma', alpha=0.45)
    axes[2].set_title('Integrated Gradients overlay')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(NOTE, fontsize=10)
    plt.tight_layout()
    plt.show()

## Expected output

The notebook saves Grad-CAM and Integrated Gradients heatmaps, overlays, raw attribution arrays, and `xai_summary.csv` under `results/xai/`. Attribution maps show regions influential for the model prediction, not direct dark matter detections.